<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/02_suma_escalar_y_producto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 02 &middot; Suma, producto por escalar y multiplicación

**Módulo 1 — Álgebra lineal y geometría diferencial**

Sumar, escalar y multiplicar: son las tres operaciones con las que se construye absolutamente
todo lo demás. Entrenar una red neuronal es, literalmente, repetir millones de veces
`salida = entrada @ pesos + sesgo` y después `pesos = pesos - tasa * gradiente`. Ahí no hay
más que una multiplicación de matrices, una suma y un producto por escalar.

Este notebook las trabaja una por una, con la regla escrita a mano *y* con NumPy, para que
cuando veas `A @ B` sepas exactamente qué está pasando dentro.

## Al terminar vas a poder

- **Sumar** vectores y matrices, y saber cuándo la operación no está definida.
- Multiplicar por un **escalar** y leer qué le hace eso a la geometría.
- Distinguir las **tres cosas distintas** que la gente llama "multiplicar dos vectores":
  producto punto, producto elemento a elemento y producto externo.
- Multiplicar **matrices** a mano siguiendo la regla fila × columna, y verificarlo con NumPy.
- Explicar por qué $AB \neq BA$ y por qué eso importa al apilar capas de una red.

## Qué necesitas saber antes

El notebook [01 · Operaciones básicas](01_operaciones_vectores.ipynb) (producto punto y norma)
y saber leer código sencillo de Python. Si no lo has visto, éste se entiende igual: el producto
punto se vuelve a explicar en la sección 3.

## Cómo usar este notebook

1. Si entraste con el botón **Open in Colab** de arriba, no tienes que instalar nada.
2. Ejecuta las celdas en orden con `Shift + Enter`.
3. **Cambia los números y vuelve a ejecutar.** Ahí es donde de verdad se aprende.
4. Al final hay una sección **Tu turno** con ejercicios para resolver.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Menos decimales y sin notacion cientifica: los resultados se leen mejor.
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5, 5)

print("numpy:", np.__version__)

---

## 1. Suma de vectores

Se suman **componente con componente**:

$$\vec{u} + \vec{v} = (u_1 + v_1,\; u_2 + v_2,\; \dots,\; u_n + v_n)$$

Sólo hay una condición: los dos vectores deben tener **el mismo número de componentes**. No
existe la suma de un vector de 2 con uno de 3; NumPy lanza un error y hace bien.

Geométricamente es la **regla del paralelogramo**: recorres $\vec{u}$, y desde donde quedaste
recorres $\vec{v}$. Donde terminas es $\vec{u} + \vec{v}$.

In [ ]:
u = np.array([3.0, 1.0])
v = np.array([1.0, 4.0])

# A mano, para ver que no hay magia:
suma_manual = np.array([u[i] + v[i] for i in range(len(u))])

print("u       =", u)
print("v       =", v)
print("a mano  =", suma_manual)
print("numpy   =", u + v)
print("iguales :", np.allclose(suma_manual, u + v))

# La resta es lo mismo con signo contrario: u - v = u + (-1)*v
print("\nu - v   =", u - v)

# Y cuando las dimensiones no coinciden, la operacion no existe:
try:
    u + np.array([1.0, 2.0, 3.0])
except ValueError as e:
    print("\nsumar (2,) con (3,) ->", e)

In [ ]:
def dibujar_vectores(vectores, etiquetas, limite=6, titulo="", desde=None):
    """Dibuja vectores como flechas.

    `desde` permite que una flecha no salga del origen, que es lo que se
    necesita para ver la regla del paralelogramo.
    """
    colores = plt.cm.tab10.colors
    origenes = desde if desde is not None else [(0, 0)] * len(vectores)
    fig, ax = plt.subplots()
    for i, (vec, nombre, o) in enumerate(zip(vectores, etiquetas, origenes)):
        ax.quiver(o[0], o[1], vec[0], vec[1], angles="xy", scale_units="xy", scale=1,
                  color=colores[i % len(colores)], width=0.012)
        ax.text(o[0] + vec[0] * 0.55, o[1] + vec[1] * 0.55, nombre,
                color=colores[i % len(colores)], fontsize=12, fontweight="bold")
    ax.set_xlim(-limite, limite); ax.set_ylim(-limite, limite)
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3); ax.set_aspect("equal"); ax.set_title(titulo)
    plt.show()


# u, luego v desde la punta de u: se llega justo a u+v.
dibujar_vectores(
    [u, v, u + v],
    ["u", "v", "u+v"],
    limite=7,
    titulo="Regla del paralelogramo",
    desde=[(0, 0), (u[0], u[1]), (0, 0)],
)

---

## 2. Multiplicación por un escalar

Un **escalar** es un número suelto, no un vector. Multiplicar por él estira o encoge la flecha
sin cambiar la recta sobre la que vive:

$$c\,\vec{u} = (c\,u_1,\; c\,u_2,\; \dots,\; c\,u_n)$$

- $c > 1$ la alarga, $0 < c < 1$ la acorta.
- $c < 0$ además la **voltea** al lado contrario.
- $c = 0$ la aplasta al origen.

De ahí viene el nombre: el escalar *escala*. Y fíjate que $\|c\,\vec{u}\| = |c|\,\|\vec{u}\|$:
la longitud se multiplica por el valor absoluto de $c$.

In [ ]:
w = np.array([2.0, 1.0])

for c in [2.0, 0.5, -1.5]:
    print(f"c = {c:>5} -> c*w = {c * w}   norma = {np.linalg.norm(c * w):.4f}")

print(f"\nnorma de w   = {np.linalg.norm(w):.4f}")
print(f"norma de 2*w = {np.linalg.norm(2 * w):.4f}  <- exactamente el doble")

dibujar_vectores([2 * w, w, 0.5 * w, -1.5 * w],
                 ["2w", "w", "0.5w", "-1.5w"],
                 limite=5, titulo="El escalar estira, encoge o voltea")

### Suma y escalar juntos: la combinación lineal

Casi todo el álgebra lineal es esto: escalar unos vectores y sumarlos.

$$c_1\vec{v}_1 + c_2\vec{v}_2 + \dots + c_k\vec{v}_k$$

**En IA lo ves en el paso de entrenamiento.** El descenso de gradiente actualiza los pesos así:

$$\vec{w}_{\text{nuevo}} = \vec{w} - \eta\,\nabla L$$

Un producto por escalar (la tasa de aprendizaje $\eta$ multiplicando al gradiente) y una resta
de vectores. Toda la magia del entrenamiento cabe en esa línea, repetida miles de veces.

In [ ]:
def paso_de_descenso(w, gradiente, tasa):
    """Un paso de descenso de gradiente: escalar el gradiente y restarlo."""
    return w - tasa * gradiente


pesos = np.array([1.0, -2.0, 0.5])
grad = np.array([0.4, 0.1, -0.3])     # lo que devolveria la retropropagacion
tasa = 0.1                            # el famoso learning rate

print("pesos antes   :", pesos)
print("gradiente     :", grad)
print("pesos despues :", paso_de_descenso(pesos, grad, tasa))

# Con una tasa enorme el paso se pasa de largo: por eso eta suele ser pequena.
print("con tasa 5.0  :", paso_de_descenso(pesos, grad, 5.0), "<- se disparo")

---

## 3. "Multiplicar dos vectores" son tres cosas distintas

Aquí está la confusión más común de todo el tema. Con dos vectores del mismo tamaño puedes
hacer al menos tres productos diferentes, y **cada uno devuelve algo de forma distinta**:

| Nombre | Notación | Qué devuelve | En NumPy |
|---|---|---|---|
| Producto punto (escalar) | $\vec{u}\cdot\vec{v}$ | **un número** | `np.dot(u, v)` o `u @ v` |
| Elemento a elemento (Hadamard) | $\vec{u}\odot\vec{v}$ | **un vector** del mismo tamaño | `u * v` |
| Producto externo | $\vec{u}\,\vec{v}^{\,T}$ | **una matriz** $n \times m$ | `np.outer(u, v)` |

$$\vec{u}\cdot\vec{v} = \sum_i u_i v_i \qquad
(\vec{u}\odot\vec{v})_i = u_i v_i \qquad
(\vec{u}\,\vec{v}^{\,T})_{ij} = u_i v_j$$

> **Ojo con esto:** en NumPy el asterisco `*` **no** es el producto punto, es el elemento a
> elemento. Es el error silencioso más frecuente al empezar, porque no truena: simplemente
> devuelve algo distinto de lo que querías.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 0.0, -1.0])

punto = np.dot(a, b)          # = 1*4 + 2*0 + 3*(-1)
hadamard = a * b              # = [1*4, 2*0, 3*(-1)]
externo = np.outer(a, b)      # matriz 3x3

print("a =", a)
print("b =", b)
print("\nproducto punto      :", punto, "forma:", np.shape(punto), "(un escalar)")
print("elemento a elemento :", hadamard, "forma:", hadamard.shape)
print("producto externo    : forma:", externo.shape)
print(externo)

# El punto es la SUMA del elemento a elemento: esa es la relacion entre los dos primeros.
print("\nsuma del Hadamard   :", hadamard.sum(),
      "== producto punto:", np.isclose(hadamard.sum(), punto))

### Para qué sirve cada uno en IA

- **Producto punto**: es lo que calcula una neurona, $f(\vec{w}\cdot\vec{x} + b)$. También es
  la similitud coseno con la que un buscador semántico compara textos.
- **Elemento a elemento**: son las *compuertas*. Un LSTM decide qué recordar multiplicando su
  estado por un vector de valores entre 0 y 1; *dropout* apaga neuronas al azar multiplicando
  por un vector de ceros y unos. Multiplicar por 0 es apagar, por 1 es dejar pasar.
- **Producto externo**: construye una matriz a partir de dos vectores. Aparece en la
  retropropagación (el gradiente respecto a los pesos de una capa tiene justamente esa forma)
  y en las aproximaciones de rango bajo, como LoRA para ajustar modelos grandes.

In [ ]:
activaciones = np.array([0.8, -1.2, 2.0, 0.5, -0.3])

# Dropout: apagar la mitad de las neuronas multiplicando por una mascara de 0 y 1.
mascara = np.array([1.0, 0.0, 1.0, 0.0, 1.0])
print("activaciones :", activaciones)
print("mascara      :", mascara)
print("tras dropout :", activaciones * mascara, "<- elemento a elemento")

# Compuerta suave: en vez de 0/1, valores intermedios que dejan pasar 'a medias'.
compuerta = np.array([0.9, 0.1, 0.5, 0.0, 1.0])
print("\ncon compuerta:", activaciones * compuerta)

# Producto externo: la matriz que aparece al derivar respecto a los pesos.
entrada = np.array([1.0, 2.0])
delta = np.array([0.5, -1.0, 0.25])       # error que llega de la capa siguiente
print("\ngradiente de W (delta por entrada):")
print(np.outer(delta, entrada), "forma:", np.outer(delta, entrada).shape)

---

## 4. Suma de matrices

Igual que con vectores: **elemento a elemento**, y las dos matrices deben tener exactamente
**la misma forma**.

$$(A + B)_{ij} = a_{ij} + b_{ij}$$

Una matriz $3\times 2$ no se puede sumar con una $2\times 3$, aunque tengan los mismos seis
números. La forma manda.

In [ ]:
A = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])

B = np.array([[10.0, 20.0],
              [30.0, 40.0],
              [50.0, 60.0]])

print("A (forma", A.shape, ")\n", A)
print("\nB (forma", B.shape, ")\n", B)
print("\nA + B =\n", A + B)
print("\nA - B =\n", A - B)

try:
    A + A.T                      # (3,2) + (2,3): no existe
except ValueError as e:
    print("\nsumar (3,2) con (2,3) ->", e)

### Broadcasting: sumar un vector a todas las filas

NumPy hace una excepción muy útil: si sumas una matriz $n\times m$ con un vector de $m$
componentes, **repite el vector en cada fila**. Se llama *broadcasting*.

No es una regla del álgebra lineal de pizarrón, es una comodidad de NumPy, pero se usa
constantemente: así se le suma el **sesgo** $\vec{b}$ a todas las muestras de un lote de datos
de una sola vez, sin escribir un ciclo.

In [ ]:
lote = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0],
                 [7.0, 8.0, 9.0],
                 [0.0, 1.0, 0.0]])       # 4 muestras, 3 caracteristicas cada una

sesgo = np.array([100.0, 200.0, 300.0])  # un sesgo por caracteristica

print("lote (forma", lote.shape, ")\n", lote)
print("\nsesgo (forma", sesgo.shape, ") =", sesgo)
print("\nlote + sesgo  <- el sesgo se repite en las 4 filas\n", lote + sesgo)

---

## 5. Matriz por un escalar

Se multiplica **cada entrada** por el número:

$$(cA)_{ij} = c\,a_{ij}$$

Sirve para cambiar la escala de todos los pesos a la vez. Es lo que ocurre, por ejemplo, cuando
aplicas *weight decay* (regularización) o cuando normalizas una imagen dividiendo sus píxeles
entre 255 para llevarlos al rango $[0, 1]$.

In [ ]:
print("A =\n", A)
print("\n3 * A =\n", 3 * A)
print("\n0.5 * A =\n", 0.5 * A)
print("\n-A =\n", -A, "\n<- es (-1) * A")

# Caso tipico: pasar una imagen de 0-255 a 0-1 dividiendo entre un escalar.
pixeles = np.array([[0.0, 128.0, 255.0],
                    [64.0, 200.0, 32.0]])
print("\npixeles / 255 =\n", pixeles / 255)

---

## 6. Multiplicación de matrices

Ésta es la única que **no** es elemento a elemento, y es la que de verdad importa.

$$C = AB \qquad c_{ij} = \sum_{k} a_{ik}\,b_{kj}$$

En palabras: la entrada $c_{ij}$ es el **producto punto de la fila $i$ de $A$ con la columna
$j$ de $B$**. Toda la multiplicación de matrices son productos punto, uno por cada casilla del
resultado.

**La regla de las formas**, que es lo primero que hay que revisar siempre:

$$\underbrace{A}_{n \times m} \cdot \underbrace{B}_{m \times p} = \underbrace{C}_{n \times p}$$

Las dimensiones de en medio tienen que **coincidir y se cancelan**; las de los extremos son la
forma del resultado. Si no coinciden, el producto no existe.

In [ ]:
def multiplicar_a_mano(A, B):
    """Producto matricial siguiendo la definicion, sin usar @."""
    n, m = A.shape
    m2, p = B.shape
    if m != m2:
        raise ValueError(f"no se puede: {A.shape} por {B.shape}")

    C = np.zeros((n, p))
    for i in range(n):                 # cada fila del resultado
        for j in range(p):             # cada columna del resultado
            # producto punto de la fila i de A con la columna j de B
            for k in range(m):
                C[i, j] += A[i, k] * B[k, j]
    return C


M = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])        # 2x3

N = np.array([[7.0, 8.0],
              [9.0, 10.0],
              [11.0, 12.0]])           # 3x2

print("M forma", M.shape, " N forma", N.shape, " -> M@N forma (2, 2)")
print("\na mano =\n", multiplicar_a_mano(M, N))
print("\nnumpy  =\n", M @ N)
print("\niguales:", np.allclose(multiplicar_a_mano(M, N), M @ N))

# Paso a paso una sola casilla, para verla clarita:
print("\nc[0,0] = fila 0 de M . columna 0 de N")
print(f"       = {M[0]} . {N[:, 0]}")
print(f"       = {M[0,0]}*{N[0,0]} + {M[0,1]}*{N[1,0]} + {M[0,2]}*{N[2,0]} = {M[0] @ N[:, 0]}")

In [ ]:
# Cuando las formas no encajan, no hay producto:
try:
    M @ M                    # (2,3) @ (2,3): el 3 y el 2 de en medio no coinciden
except ValueError as e:
    print("(2,3) @ (2,3) ->", e)

# Pero M @ M.T si, porque (2,3) @ (3,2) -> (2,2)
print("\nM @ M.T =\n", M @ M.T, "\nforma:", (M @ M.T).shape)
print("\nM.T @ M =\n", M.T @ M, "\nforma:", (M.T @ M).shape, " <- otra forma, otro resultado")

### Propiedades: qué se vale y qué no

Con $A$, $B$, $C$ matrices de formas compatibles:

| Propiedad | ¿Se cumple? |
|---|---|
| $AB = BA$ (conmutativa) | **No** |
| $(AB)C = A(BC)$ (asociativa) | Sí |
| $A(B + C) = AB + AC$ (distributiva) | Sí |
| $AI = IA = A$ (identidad) | Sí |

La primera es la importante: **el orden no da igual**. Aplicar una rotación y luego un
estiramiento no es lo mismo que estirar y luego rotar. En una red neuronal esto significa que
el orden de las capas es parte del modelo, no un detalle.

In [ ]:
R = np.array([[0.0, -1.0],       # rotacion de 90 grados
              [1.0,  0.0]])

S = np.array([[3.0, 0.0],        # estirar 3x en horizontal
              [0.0, 1.0]])

print("R @ S =\n", R @ S)
print("\nS @ R =\n", S @ R)
print("\nson iguales?", np.allclose(R @ S, S @ R), " <- el orden SI importa")

# Asociativa y distributiva si se cumplen:
T = np.array([[1.0, 2.0], [0.0, 1.0]])
print("\n(R@S)@T == R@(S@T) :", np.allclose((R @ S) @ T, R @ (S @ T)))
print("R@(S+T) == R@S + R@T:", np.allclose(R @ (S + T), R @ S + R @ T))

# La identidad es el 1 de las matrices:
I = np.eye(2)
print("\nI =\n", I)
print("R @ I == R:", np.allclose(R @ I, R))

In [ ]:
# Verlo con figuras: el mismo cuadrado, las mismas dos operaciones, distinto orden.
cuadrado = np.array([[0, 1, 1, 0, 0],
                     [0, 0, 1, 1, 0]])       # 4 esquinas, cerrando en la primera

fig, ejes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (Mat, titulo) in zip(ejes, [(R @ S, "R@S: estirar y luego rotar"),
                                    (S @ R, "S@R: rotar y luego estirar")]):
    resultado = Mat @ cuadrado
    ax.plot(cuadrado[0], cuadrado[1], "o-", color="tab:blue", label="original")
    ax.fill(cuadrado[0], cuadrado[1], alpha=0.2, color="tab:blue")
    ax.plot(resultado[0], resultado[1], "o-", color="tab:red", label="transformado")
    ax.fill(resultado[0], resultado[1], alpha=0.2, color="tab:red")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3); ax.set_aspect("equal"); ax.set_title(titulo); ax.legend()
plt.show()

---

## 7. Una capa de red neuronal es exactamente esto

Junta todo lo del notebook y tienes una capa densa completa:

$$Y = XW + \vec{b}$$

- $X$ es el **lote de datos**: una fila por muestra, una columna por característica.
- $W$ son los **pesos**: una columna por neurona de salida.
- $\vec{b}$ es el **sesgo**, que se suma a todas las filas por broadcasting.

Multiplicación de matrices, suma de vector, y ya. Que un modelo tenga miles de millones de
parámetros no cambia la operación: sólo cambian las formas.

In [ ]:
def capa_densa(X, W, b):
    """Una capa densa con activacion ReLU."""
    Z = X @ W + b                      # producto de matrices + suma con broadcasting
    return np.maximum(0.0, Z)          # ReLU, elemento a elemento


np.random.seed(0)
X = np.array([[0.5, 2.0, -1.0],
              [1.0, 0.0, 3.0],
              [-2.0, 1.5, 0.5],
              [0.0, -1.0, 1.0]])       # 4 muestras x 3 caracteristicas

W = np.random.randn(3, 2)              # 3 entradas -> 2 neuronas
b = np.array([0.1, -0.2])              # un sesgo por neurona

print("X:", X.shape, " W:", W.shape, " b:", b.shape)
print("-> salida esperada: (4, 2)\n")
salida = capa_densa(X, W, b)
print("salida =\n", salida, "\nforma:", salida.shape)

# Dos capas encadenadas: la salida de una es la entrada de la siguiente.
W2 = np.random.randn(2, 1)
b2 = np.array([0.0])
print("\nsalida de la segunda capa =\n", capa_densa(salida, W2, b2))

### Por qué las GPU

Cuenta las operaciones: para multiplicar una $n\times m$ por una $m\times p$ hacen falta
$n \cdot m \cdot p$ multiplicaciones y otras tantas sumas. Con matrices de $1000 \times 1000$
eso ya son **mil millones** de multiplicaciones para *un solo* producto.

Lo importante es que todas esas casillas son independientes entre sí: se pueden calcular al
mismo tiempo. Por eso el hardware que entrena modelos son GPU, hechas para hacer miles de
multiplicaciones en paralelo. Y por eso, cuando programas, dejas que NumPy haga el producto en
vez de escribir los ciclos tú.

In [ ]:
import time

n = 120
P = np.random.rand(n, n)
Q = np.random.rand(n, n)

t0 = time.perf_counter()
lento = multiplicar_a_mano(P, Q)          # tres ciclos de Python
t_lento = time.perf_counter() - t0

t0 = time.perf_counter()
rapido = P @ Q                            # NumPy por debajo llama a BLAS, escrito en C
t_rapido = time.perf_counter() - t0

print(f"multiplicaciones necesarias: {n ** 3:,}")
print(f"con ciclos de Python : {t_lento:.4f} s")
print(f"con @ de NumPy       : {t_rapido:.6f} s")
print(f"NumPy es ~{t_lento / max(t_rapido, 1e-9):,.0f} veces mas rapido")
print("mismo resultado:", np.allclose(lento, rapido))

---

## Tu turno

Resuelve en la celda de abajo. A propósito no hay respuestas al final: la idea es que
**verifiques cada resultado con código**, comparando tu versión contra la de NumPy.

1. Con $\vec{p} = (2, -1, 4)$ y $\vec{q} = (0, 3, 1)$: calcula $\vec{p}+\vec{q}$,
   $3\vec{p} - 2\vec{q}$, el producto punto, el elemento a elemento y el producto externo.
   Anota la **forma** de cada resultado antes de ejecutar y luego comprueba si acertaste.
2. Sin ejecutar nada, di qué forma tiene el resultado de cada producto, o si no existe:
   $(4\times3)(3\times5)$, $(2\times2)(2\times2)$, $(3\times1)(1\times3)$, $(5\times2)(5\times2)$.
   Después verifícalo creando las matrices con `np.zeros` y mirando `.shape`.
3. Escribe `mi_producto(A, B)` usando **sólo** `np.dot` sobre filas y columnas (un ciclo doble,
   sin el tercero) y comprueba con `np.allclose` que coincide con `A @ B`.
4. Encuentra dos matrices $2\times2$ que **sí** conmuten ($AB = BA$) y que ninguna sea la
   identidad ni un múltiplo de ella. Pista: prueba con potencias de una misma matriz.
5. Comprueba con código que $(AB)^T = B^T A^T$ para dos matrices de formas compatibles.
   ¿Por qué se invierte el orden?
6. Pásale a `capa_densa` un lote de **una sola muestra** con `X[0:1]` y compara la salida con
   `capa_densa(X, W, b)[0]`. ¿Por qué hay que escribir `X[0:1]` y no `X[0]`?

In [ ]:
# Tu codigo aqui.
p = np.array([2.0, -1.0, 4.0])
q = np.array([0.0, 3.0, 1.0])

# 1.


---

## Resumen

| Operación | En NumPy | Condición | Resultado |
|---|---|---|---|
| Suma de vectores | `u + v` | misma dimensión | vector |
| Vector por escalar | `c * u` | siempre | vector |
| Producto punto | `np.dot(u, v)` o `u @ v` | misma dimensión | **escalar** |
| Elemento a elemento | `u * v` | misma dimensión | vector |
| Producto externo | `np.outer(u, v)` | siempre | **matriz** $n\times m$ |
| Suma de matrices | `A + B` | misma forma | matriz igual |
| Matriz por escalar | `c * A` | siempre | matriz igual |
| Producto de matrices | `A @ B` | $(n\times m)(m\times p)$ | matriz $n\times p$ |

Tres ideas para llevarte:

1. **Suma, resta, escalar y Hadamard son elemento a elemento.** El producto de matrices no: es
   fila por columna.
2. **En NumPy `*` no es el producto punto.** `*` es elemento a elemento; `@` es el producto
   matricial.
3. **Revisa siempre las formas antes que los números.** La mayoría de los errores en código de
   IA son formas que no encajan, no matemáticas equivocadas.

## Qué sigue

- La versión en script, más corta y sin explicaciones: [`02_suma_escalar_y_producto.py`](02_suma_escalar_y_producto.py)
- El notebook anterior: [`01 · Operaciones básicas con vectores y matrices`](01_operaciones_vectores.ipynb)
- Los demás módulos, en el [README del repositorio](../../README.md)

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*